In [1]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


# Full pKₐ Prediction Pipeline

This notebook implements:
1. Data loading & salt removal
2. 2D/3D & quantum‐chemical descriptors (RDKit, Mordred, QCT)
3. Pretrained GNN (MolGpKa) with transfer learning
4. Bayesian HPO (Optuna) for XGBoost, GNN, and ensemble
5. Nested CV grid search for QLattice symbolic regression with multiple seeds
6. Ensemble of best symbolic formulas
7. Final evaluation and printing of formulas & stats

In [2]:
############# 1. Imports #############
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import DataLoader, InMemoryDataset, Data
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Fragments, SaltRemover
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import optuna
import feyn
from itertools import product
from tqdm import tqdm
import logging
print('✅ Libraries imported')

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

✅ Libraries imported


In [3]:
############# 2. Load & Clean Data #############
train_df = pd.read_csv('./dw_data/Opt2_acidic_tr.csv')
test_df  = pd.read_csv('./dw_data/Opt2_acidic_tst.csv')
target = 'pKa'
for df in [train_df, test_df]:
    df[target] = pd.to_numeric(df[target], errors='coerce')
saltRemover = SaltRemover.SaltRemover()
for df in [train_df, test_df]:
    df['Mol'] = df['OriginalSmiles'].astype(str).apply(lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s)))
print('✅ Data loaded and salts removed')

✅ Data loaded and salts removed


In [4]:
############# 3. 2D/3D & Mordred Descriptors #############
# --- Descriptor Computation (RDKit + Mordred only) ---
calc = Calculator(descriptors, ignore_3D=True)

def compute_descriptors(df):
    mols = df['Mol'].tolist()
    # RDKit 2D descriptors
    rdkit_list = []
    for m in tqdm(mols, desc="RDKit Descriptors"):
        rdkit_list.append({name: fn(m) for name, fn in Descriptors.descList})
    rdkit_df = pd.DataFrame(rdkit_list)
    # Fragment counts
    frag_list = []
    for m in tqdm(mols, desc="Fragment Counts"):
        frag_list.append({name: fn(m) for name, fn in Fragments.__dict__.items() if callable(fn) and name.startswith("fr_")})
    frag_df = pd.DataFrame(frag_list)
    # Mordred descriptors
    mordred_df = calc.pandas(mols)
    # Combine and clean
    full = pd.concat([df[[target]].reset_index(drop=True), rdkit_df, frag_df, mordred_df], axis=1)
    # Drop any all-NaN or zero-variance columns
    full = full.dropna(axis=1, how="all")
    non_zero = full.loc[:, full.std(numeric_only=True) > 0]
    return non_zero.dropna()

# Then call:
train_desc = compute_descriptors(train_df)
test_desc  = compute_descriptors(test_df)
print("✅ Descriptors computed (RDKit + Mordred).")


RDKit Descriptors:   1%|▋                                                           | 26/2321 [00:00<00:09, 249.56it/s]


ValueError: max() arg is an empty sequence

In [ ]:
############# 4. Pretrained GNN (MolGpKa) & Transfer Learning #############
class MolGpKa(torch.nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.conv1 = GCNConv(ATOM_DIM, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin   = torch.nn.Linear(hidden, 1)
    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)
# Load or define dataset for pretraining
# ... code to load large pKa dataset into PyG format ...
# Pretrain then fine-tune on our train_desc set
print('✅ MolGpKa model defined (pretraining code omitted)')

In [ ]:
############# 5. Feature Alignment & Scaling #############
def sanitize(df):
    df = df.loc[:, df.columns.notna()]
    df = df.loc[:, df.notna().all()]
    df.columns = [str(c) for c in df.columns]
    return df.loc[:,~df.columns.duplicated()].reset_index(drop=True)
train_desc = sanitize(train_desc)
test_desc  = sanitize(test_desc)
X_train = train_desc.drop(columns=[target])
X_test  = test_desc .drop(columns=[target])
# align columns
for c in set(X_train)-set(X_test): X_test[c]=0
X_test = X_test[X_train.columns]
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
y_train = train_desc[target].values
y_test  = test_desc [target].values
print('✅ Data aligned & scaled')

In [ ]:
############# 6. Optuna HPO for XGBoost & GNN #############
def objective_xgb(trial):
    params = {
      'max_depth': trial.suggest_int('max_depth', 3,10),
      'learning_rate': trial.suggest_loguniform('lr',1e-3,1e-1),
      'n_estimators': trial.suggest_int('n_estimators',100,500),
      'subsample': trial.suggest_uniform('sub',0.6,1.0),
      'colsample_bytree': trial.suggest_uniform('col',0.6,1.0)
    }
    model = XGBRegressor(**params, tree_method='hist')
    kf = KFold(3, shuffle=True, random_state=42)
    scores = []
    for tr,va in kf.split(X_train):
        model.fit(X_train[tr], y_train[tr])
        preds = model.predict(X_train[va])
        scores.append(r2_score(y_train[va],preds))
    return np.mean(scores)
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30)
best_xgb = XGBRegressor(**study_xgb.best_params)
best_xgb.fit(X_train,y_train)
print('✅ XGBoost HPO done:', study_xgb.best_params)

In [ ]:
############# 7. Baseline Ensemble #############
from sklearn.ensemble import StackingRegressor
estimators = [
  ('xgb', best_xgb),
  ('rf', RandomForestRegressor(n_estimators=200, random_state=42)),
  ('lgb', lgb.LGBMRegressor(n_estimators=200, random_state=42))
]
stack = StackingRegressor(estimators=estimators, final_estimator=XGBRegressor())
stack.fit(X_train,y_train)
preds = stack.predict(X_test)
print('Ensemble R²:', r2_score(y_test,preds))
print('✅ Baseline ensemble built')

In [ ]:
############# 8. Nested CV QLattice Grid Search w/ Multiple Seeds #############
ql = feyn.QLattice()
param_grid = {
  'n_epochs': [100,200,300],
  'max_complexity': [150,200,250],
  'criterion': ['aic','bic']
}
combs = list(product(param_grid['n_epochs'],param_grid['max_complexity'],param_grid['criterion'], range(3)))
# (last dim is seed index)
results=[]
kf=KFold(3,shuffle=True,random_state=42)
for epochs,comp,crit,seed in tqdm(combs,desc='QLattice Search'):
    cv_scores=[]
    for tr,va in kf.split(train_desc):
        tr_df = train_desc.iloc[tr][[target]+top_features]
        va_df = train_desc.iloc[va][[target]+top_features]
        models = ql.auto_run(tr_df, output_name=target,
                             n_epochs=epochs,threads=16,
                             max_complexity=comp,criterion=crit,seed=seed)
        if models:
            cv_scores.append(r2_score(va_df[target],models[0].predict(va_df)))
    if cv_scores:
        results.append({'epochs':epochs,'comp':comp,'crit':crit,'seed':seed,'cv_r2':np.mean(cv_scores)})
best = pd.DataFrame(results).sort_values('cv_r2',ascending=False).iloc[0]
print('Best QLattice params:',best)
print('✅ Nested CV QLattice search complete')

In [ ]:
############# 9. Retrain Best QLattice & Print Formula/Stats #############
params = best.to_dict()
models = ql.auto_run(train_desc[[target]+top_features],
                    output_name=target,
                    n_epochs=int(params['epochs']),
                    threads=16,
                    max_complexity=int(params['comp']),
                    criterion=params['crit'],
                    seed=int(params['seed']))
model = models[0]
preds = model.predict(test_desc[[target]+top_features])
print('Final QLattice Test R²:',r2_score(test_desc[target],preds))
try:
    print('\n## Formula:')
    print(model.formula())
except:
    print(model.pretty())
print('\n## Statistics:')
print(model.stats())
